# Tráfico en Madrid: Entrenamiento y optimización de modelos

Llegamos al notebook donde ponemos a trabajar los datos. El NB02 dejó los splits ya preparados en `data/final/` como parquets, así que aquí no tocamos limpieza: entrenamos tres familias de modelos distintas, comparamos cómo rinden, cogemos el mejor y le ajustamos los hiperparámetros. El resultado final es un único pipeline serializado que el NB04 cargará para la evaluación profunda.

La parte importante no es tanto sacar el mejor F1 posible como justificar cada elección: qué métrica miramos y por qué, qué algoritmos tienen sentido con millones de filas, qué transformaciones necesita cada modelo y cómo optimizamos sin que la búsqueda tarde horas.


# 1. Configuración del entorno

Dejamos al principio los imports, constantes y configuración visual para tener un único sitio donde tocar cualquier parámetro. Fijamos `RANDOM_STATE = 42` en todos los splits, modelos y muestreos para que la ejecución sea reproducible.


In [1]:
%pip install -r ../requirements.txt -q

Note: you may need to restart the kernel to use updated packages.


Imports en el orden estándar: librería estándar, terceros, y separados por una línea en blanco. Solo traemos lo que vamos a usar.

In [2]:
# Librería estándar
import time
import json
from pathlib import Path

# Terceros
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

Constantes del notebook. Las rutas vienen de `data/final/`, que es donde el NB02 guardó los parquets. `FEATURES_MODELO` es la lista de columnas que entran al modelo: hemos quitado `id` porque son 4652 sensores únicos y como variable numérica ordinal no aporta nada útil. La localización ya la dan `utm_x`, `utm_y`, `distrito` y `tipo_elem`.

Agrupamos las columnas por tipo porque el pipeline de regresión logística las tratará de forma distinta: las cíclicas con sin/cos, las continuas con StandardScaler, la categórica con one-hot y las binarias tal cual.


In [3]:
RANDOM_STATE = 42

DATA_FINAL = Path("../data/final")
RUTA_MODELO = DATA_FINAL / "modelo_congestion.joblib"
RUTA_METADATOS = DATA_FINAL / "modelo_metadatos.json"

FEATURES_MODELO = [
    "tipo_elem",
    "distrito",
    "utm_x",
    "utm_y",
    "hora",
    "dia_semana",
    "es_laborable",
    "es_hora_punta",
]
COLS_CICLICAS = ["hora", "dia_semana"]
COLS_CONTINUAS = ["utm_x", "utm_y"]
COL_CATEGORICA = ["distrito"]
COLS_PASSTHROUGH = ["tipo_elem", "es_laborable", "es_hora_punta"]

# Tamaño de la muestra estratificada para la búsqueda de hiperparámetros.
# Está fijo aposta: si el train crece a un año entero, la búsqueda sigue siendo
# abordable porque siempre se trabaja sobre 2M filas.
TAM_MUESTRA_BUSQUEDA = 2_000_000
TAM_MUESTRA_IMPORTANCIAS = 100_000

Ajustes de pandas y matplotlib idénticos a NB01/NB02 para que las tablas y gráficas mantengan coherencia visual.

In [4]:
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

sns.set_theme(style="ticks", palette="muted", font_scale=0.9)
plt.rcParams.update(
    {
        "figure.figsize": (10, 4),
        "figure.dpi": 100,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titleweight": "normal",
        "axes.titlelocation": "left",
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.frameon": False,
        "axes.grid": True,
        "grid.linestyle": ":",
        "grid.alpha": 0.35,
    }
)

_pal = sns.color_palette("muted")
COLOR_AZUL = _pal[0]
COLOR_NARANJA = _pal[1]
COLOR_VERDE = _pal[2]
COLOR_ROJO = _pal[3]

## 1.1 Carga de los datos

Leemos los tres splits (train, validación, test) que el NB02 dejó en `data/final/`. El holdout `eval_final` no lo tocamos aquí: es para la evaluación definitiva del NB04. Quitamos la columna `id` porque no entra al modelo.


In [5]:
def cargar_split(nombre: str):
    X = pd.read_parquet(DATA_FINAL / f"X_{nombre}.parquet")
    y = pd.read_parquet(DATA_FINAL / f"y_{nombre}.parquet").squeeze("columns")
    X = X.drop(columns=["id"])
    return X[FEATURES_MODELO], y


X_train, y_train = cargar_split("train")
X_val, y_val = cargar_split("val")
X_test, y_test = cargar_split("test")

print(f"Train: {X_train.shape[0]:>10,} filas — positivos: {y_train.mean():.2%}")
print(f"Val:   {X_val.shape[0]:>10,} filas — positivos: {y_val.mean():.2%}")
print(f"Test:  {X_test.shape[0]:>10,} filas — positivos: {y_test.mean():.2%}")
print(f"\nFeatures que entran al modelo ({len(FEATURES_MODELO)}):")
print(list(X_train.columns))

Train: 20,153,118 filas — positivos: 4.53%
Val:    2,519,141 filas — positivos: 4.53%
Test:   2,519,141 filas — positivos: 4.53%

Features que entran al modelo (8):
['tipo_elem', 'distrito', 'utm_x', 'utm_y', 'hora', 'dia_semana', 'es_laborable', 'es_hora_punta']


Verificamos que los tres splits tienen la misma proporción de positivos — el split del NB02 es estratificado, así que la tasa de congestión debe salir prácticamente idéntica en los tres. Si alguna se desviara mucho, sería señal de que algo ha ido mal en la preparación.


In [6]:
assert X_train.shape[0] == y_train.shape[0]
assert X_val.shape[0] == y_val.shape[0]
assert X_test.shape[0] == y_test.shape[0]
assert list(X_train.columns) == list(X_val.columns) == list(X_test.columns)
assert "id" not in X_train.columns

# 2. Estrategia de evaluación para clases desbalanceadas

Antes de entrenar nada tenemos que decidir qué métrica miramos, porque con un dataset desbalanceado al 95/5 la métrica por defecto engaña.

Si usáramos **accuracy** (porcentaje de aciertos), un modelo que predijera siempre "no congestionado" sacaría un 95.5% sin haber aprendido absolutamente nada. El problema es que ese 95.5% no detecta ni un solo atasco, que es exactamente lo que queremos predecir. La accuracy ignora la clase minoritaria, y la clase minoritaria es justo la que nos importa.

Para este problema las métricas relevantes son:

- **Recall** de la clase positiva: de todos los atascos que hubo de verdad, cuántos detectó el modelo. Es la métrica más cercana al valor de negocio — un recall bajo significa atascos no avisados.
- **Precision** de la clase positiva: de los atascos que el modelo anunció, cuántos eran reales. Un precision bajo significa falsas alarmas que saturan al usuario.
- **F1**: media armónica de recall y precision. Penaliza que cualquiera de las dos sea muy baja. Lo usamos como **métrica principal de selección del modelo ganador** porque resume el equilibrio entre detectar atascos y no inundar de falsos positivos.
- **PR-AUC** (Average Precision): área bajo la curva precision-recall. Se prefiere a ROC-AUC cuando las clases están desbalanceadas porque es más sensible al rendimiento sobre la clase minoritaria.
- **ROC-AUC**: lo calculamos como referencia pero sabemos que puede ser optimista con desbalance. Útil para comparar modelos, no para valorarlos en absoluto.

Calcularemos las cinco métricas para los tres modelos base y decidiremos el ganador por F1.


Función auxiliar que calcula las cinco métricas de un pipeline sobre un split dado. La usamos para todos los modelos base, así las comparaciones son honestas (misma función, mismo threshold de 0.5).


In [7]:
def evaluar(pipeline, X, y, nombre: str, tiempo_entreno: float = None):
    y_pred = pipeline.predict(X)
    y_proba = pipeline.predict_proba(X)[:, 1]
    return {
        "modelo": nombre,
        "accuracy": accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred),
        "recall": recall_score(y, y_pred),
        "f1": f1_score(y, y_pred),
        "pr_auc": average_precision_score(y, y_proba),
        "roc_auc": roc_auc_score(y, y_proba),
        "tiempo_entreno_s": tiempo_entreno,
    }

# 3. Selección de algoritmos y justificación

Con veinte millones de filas (y esperando crecer a un año completo) no todos los algoritmos son viables. Elegimos tres familias que cubren enfoques distintos y escalan bien, y descartamos explícitamente los que no pueden con este volumen.

**Lo que entrenamos:**

- **Regresión logística.** Baseline lineal obligatorio en cualquier problema de clasificación. Es rápida, sus coeficientes se interpretan directamente y nos dice qué parte del problema es linealmente separable. Si un modelo mucho más caro no le gana por margen claro, el baseline ya es suficiente.
- **Random Forest con `max_samples=0.1`.** Un ensamble de 200 árboles donde cada árbol se entrena sobre un 10% muestreado del train. Esta técnica de bagging con submuestreo es la forma estándar de escalar Random Forest a millones de filas: cada árbol es rápido, la diversidad entre árboles se mantiene por el bootstrap, y los 200 paralelizan con `n_jobs=-1`. Captura interacciones no lineales que la regresión logística no puede.
- **Histogram-based Gradient Boosting.** Ensamble por boosting pensado específicamente para datasets grandes. Internamente discretiza las features en histogramas de 256 bins, lo que hace que cada iteración sea órdenes de magnitud más rápida que un boosting clásico. Para datos tabulares de este tamaño suele ser el modelo con mejor rendimiento.

**Lo que descartamos y por qué:**

- **SVM.** Con kernel no lineal tiene coste `O(n²)` o peor en tiempo y memoria. Con 20M filas es inviable y con 120M imposible. El kernel lineal escala mejor pero no aporta nada sobre la regresión logística.
- **KNN.** Es perezoso: en entrenamiento no hace nada, pero en predicción calcula distancias con todos los puntos del train. Con millones de filas cada predicción tarda segundos. Además sufre la maldición de la dimensionalidad cuando añadimos one-hot de distrito (21 columnas binarias dispersas), porque las distancias se vuelven poco informativas.
- **Redes neuronales.** No aportan ventaja sobre el boosting en tabular y la complejidad justificativa no compensa para un proyecto de 3 ECTS.


# 4. Entrenamiento con pipelines y modelos base

Usamos `Pipeline` de scikit-learn para encapsular preprocesado y modelo en un único objeto. Esto tiene tres ventajas concretas: entrenamiento y predicción comparten el mismo flujo (sin fugas de datos al aplicar `.transform` por separado), el objeto serializado incluye toda la cadena de transformaciones (NB04 solo carga y predice, sin tocar preprocesado), y cualquier búsqueda de hiperparámetros puede tocar pasos del preprocesado también si hiciera falta.


## 4.1 Pipeline de regresión logística

La regresión logística necesita cuatro transformaciones sobre las features crudas:

1. **Codificación cíclica de `hora` y `dia_semana`.** El reloj es circular: las 23:00 y las 00:00 están a una hora de distancia, pero como números puros están a 23 de distancia. Si metemos `hora` como un entero de 0 a 23, el modelo lineal asume que la hora 23 es "muy distinta" de la hora 0, y aprende una pendiente espuria. La solución estándar es proyectar cada variable cíclica en dos componentes: `sin(2π·x/N)` y `cos(2π·x/N)`, donde `N` es el periodo (24 para la hora, 7 para el día de la semana). Así dos valores adyacentes en el ciclo siempre tienen coordenadas cercanas.
2. **Escalado de `utm_x` y `utm_y`.** Las coordenadas UTM son números del orden de 440.000 y 4.475.000. Sin escalado, el solver de la regresión logística tiene problemas de convergencia y los coeficientes quedan muy sesgados por la magnitud. `StandardScaler` las lleva a media 0 y desviación 1.
3. **One-hot de `distrito`.** El distrito es un código de 1 a 21 pero no hay un orden natural: el distrito 5 (Chamartín) no es "la mitad" del distrito 10 (Latina). Si dejamos el entero tal cual, la regresión logística asume una relación lineal con el código, lo cual no tiene sentido. One-hot genera 21 columnas binarias, una por distrito, y cada una recibe su propio coeficiente.
4. **Passthrough para el resto.** `tipo_elem` es binaria (0/1), `es_laborable` y `es_hora_punta` son booleanas. No hace falta transformarlas.

Los árboles del Random Forest y del HistGradientBoosting **no necesitan nada de esto**, porque internamente hacen splits discretos sobre los valores y no les afecta la escala ni la codificación ordinal.


Definimos la función de codificación cíclica a nivel de módulo. Es importante que sea una función con nombre (no un lambda) para que joblib pueda serializar el pipeline correctamente cuando lo guardemos al final.

In [8]:
def codificacion_ciclica(X):
    X = np.asarray(X, dtype=float)
    hora = X[:, 0]
    dia = X[:, 1]
    return np.column_stack([
        np.sin(2 * np.pi * hora / 24),
        np.cos(2 * np.pi * hora / 24),
        np.sin(2 * np.pi * dia / 7),
        np.cos(2 * np.pi * dia / 7),
    ])


encoder_ciclico = FunctionTransformer(codificacion_ciclica, validate=False)

Montamos el `ColumnTransformer` con las cuatro ramas y lo encadenamos con `LogisticRegression`. Usamos `class_weight='balanced'` para que el modelo penalice más los errores sobre la clase minoritaria durante el ajuste, compensando el 95/5 que tenemos en los datos. El solver `saga` escala bien a millones de filas y permite L2 por defecto.

In [ ]:
prep_logreg = ColumnTransformer(
    transformers=[
        ("ciclica", encoder_ciclico, COLS_CICLICAS),
        ("escala", StandardScaler(), COLS_CONTINUAS),
        ("onehot", OneHotEncoder(sparse_output=False, handle_unknown="ignore"), COL_CATEGORICA),
        ("passthrough", "passthrough", COLS_PASSTHROUGH),
    ],
    remainder="drop",
)

pipeline_logreg = Pipeline([
    ("prep", prep_logreg),
    ("clf", LogisticRegression(
        class_weight="balanced",
        solver="lbfgs",
        max_iter=1000,
        random_state=RANDOM_STATE,
    )),
])

Entrenamos sobre el train completo y evaluamos en val. Medimos el tiempo de entrenamiento porque es parte de la comparación: un modelo que rinde parecido pero tarda diez veces más es peor opción en la práctica.

In [10]:
t0 = time.perf_counter()
pipeline_logreg.fit(X_train, y_train)
t_logreg = time.perf_counter() - t0
print(f"Regresión logística entrenada en {t_logreg:.1f} s")

resultados_logreg = evaluar(pipeline_logreg, X_val, y_val, "LogReg", t_logreg)
pd.Series(resultados_logreg).to_frame("val").T

: 

: 

## 4.2 Pipeline de Random Forest

Random Forest no necesita preprocesado: los árboles manejan bien enteros ordinales, booleans y escalas distintas sin problemas. Mantenemos la estructura `Pipeline` por consistencia con los otros dos modelos y para que la exportación final sea homogénea.

El parámetro clave aquí es `max_samples=0.1`: cada uno de los 200 árboles entrena con un 10% muestreado (con reemplazo) del train. Para 20M filas son 2M por árbol, y escala automáticamente si el dataset crece. Sin este parámetro, Random Forest entrena cada árbol sobre el bootstrap del train entero y se vuelve inviable a partir de unos pocos millones de filas.


In [ ]:
pipeline_rf = Pipeline([
    ("clf", RandomForestClassifier(
        n_estimators=200,
        max_samples=0.1,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )),
])

t0 = time.perf_counter()
pipeline_rf.fit(X_train, y_train)
t_rf = time.perf_counter() - t0
print(f"Random Forest entrenado en {t_rf:.1f} s")

resultados_rf = evaluar(pipeline_rf, X_val, y_val, "RandomForest", t_rf)
pd.Series(resultados_rf).to_frame("val").T

## 4.3 Pipeline de Histogram-based Gradient Boosting

HistGradientBoosting es la implementación de scikit-learn del boosting con histogramas, en la línea de LightGBM. Internamente discretiza cada feature en como máximo 256 bins, lo que reduce drásticamente el coste de encontrar el mejor split en cada iteración. Es el modelo con mejor escalabilidad de los tres y suele dar los mejores resultados en tabular.

Igual que los demás, le pasamos `class_weight='balanced'` para el desbalance.


In [ ]:
pipeline_hgbm = Pipeline([
    ("clf", HistGradientBoostingClassifier(
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
])

t0 = time.perf_counter()
pipeline_hgbm.fit(X_train, y_train)
t_hgbm = time.perf_counter() - t0
print(f"HistGradientBoosting entrenado en {t_hgbm:.1f} s")

resultados_hgbm = evaluar(pipeline_hgbm, X_val, y_val, "HistGBM", t_hgbm)
pd.Series(resultados_hgbm).to_frame("val").T

# 5. Comparativa de modelos y selección

Juntamos los resultados de los tres modelos base en una tabla y una gráfica. La decisión del ganador la tomamos por **F1 en validación**, que es la métrica que mejor refleja el equilibrio entre detectar atascos y no saturar de falsas alarmas.


In [ ]:
pipelines_base = {
    "LogReg": pipeline_logreg,
    "RandomForest": pipeline_rf,
    "HistGBM": pipeline_hgbm,
}

tabla_base = pd.DataFrame([resultados_logreg, resultados_rf, resultados_hgbm])
tabla_base = tabla_base.set_index("modelo")
tabla_base

Visualizamos las cinco métricas de clasificación (dejamos fuera accuracy de la gráfica por lo que explicamos en la sección 2 — no es informativa) con barras agrupadas por modelo. Así se ve de un vistazo cuál gana en cada métrica y si hay algún modelo que sacrifica recall a cambio de precision.

In [ ]:
metricas_grafica = ["precision", "recall", "f1", "pr_auc", "roc_auc"]
df_largo = tabla_base[metricas_grafica].reset_index().melt(
    id_vars="modelo", var_name="metrica", value_name="valor"
)

fig, ax = plt.subplots(figsize=(11, 4.5))
sns.barplot(data=df_largo, x="metrica", y="valor", hue="modelo", ax=ax)
ax.set_title("Comparativa de los tres modelos base en validación")
ax.set_ylim(0, 1)
ax.set_ylabel("Valor")
ax.set_xlabel("")
ax.legend(title="Modelo", loc="upper right")
plt.tight_layout()
plt.show()

Seleccionamos el ganador por F1 en validación. Guardamos una referencia al pipeline entrenado para optimizarlo en la siguiente sección.


In [ ]:
nombre_ganador = tabla_base["f1"].idxmax()
pipeline_ganador_base = pipelines_base[nombre_ganador]
f1_ganador_base = tabla_base.loc[nombre_ganador, "f1"]

print(f"Modelo ganador por F1 en val: {nombre_ganador} (F1 = {f1_ganador_base:.4f})")

# 6. Optimización del modelo ganador

Optimizar hiperparámetros sobre los 20M (y creciendo) de filas completas con validación cruzada es inviable: un RandomizedSearch de 30 iteraciones con `cv=3` haría 90 entrenamientos sobre el dataset entero. Con el modelo más rápido estaríamos hablando de horas.

La estrategia estándar cuando el dataset es grande es hacer la búsqueda sobre una **muestra estratificada** más manejable y luego **re-entrenar el mejor modelo sobre el train completo**. Así obtenemos:

- Señal estadística suficiente para comparar combinaciones (2M filas con un 4.5% de positivos son ~90.000 casos positivos, más que de sobra para estimar F1 con precisión).
- Un ganador final ajustado a todos los datos disponibles.
- Un tiempo de búsqueda abordable (~15-30 minutos en vez de horas) que no depende de cuántos meses se hayan cargado, porque el tamaño de la muestra es fijo.

Usamos `RandomizedSearchCV` con `n_iter=30` y `cv=3`. La aleatoriedad permite explorar el espacio sin caer en la explosión combinatoria de GridSearch, y con 30 combinaciones se cubre razonablemente un espacio de 4-5 hiperparámetros.


In [ ]:
X_muestra, _, y_muestra, _ = train_test_split(
    X_train,
    y_train,
    train_size=min(TAM_MUESTRA_BUSQUEDA, len(X_train)),
    stratify=y_train,
    random_state=RANDOM_STATE,
)
print(f"Muestra para búsqueda: {len(X_muestra):,} filas — positivos: {y_muestra.mean():.2%}")

Definimos un espacio de búsqueda distinto por familia de modelo. Los nombres llevan el prefijo `clf__` porque así se referencia el paso `clf` del pipeline dentro de `RandomizedSearchCV`.

Los parámetros elegidos son los que más impactan el sesgo-varianza de cada familia:

- Para **regresión logística**, la fuerza de la regularización (`C`) y el balance de clases.
- Para **Random Forest**, la profundidad del árbol, el mínimo de muestras por hoja y el porcentaje de features por split.
- Para **HistGradientBoosting**, la tasa de aprendizaje, el número de iteraciones, el número máximo de hojas por árbol y la regularización L2.


In [ ]:
espacios_busqueda = {
    "LogReg": {
        "clf__C": [0.01, 0.1, 1.0, 10.0],
        "clf__class_weight": ["balanced", None],
    },
    "RandomForest": {
        "clf__n_estimators": [100, 200, 400],
        "clf__max_depth": [None, 10, 20, 30],
        "clf__min_samples_leaf": [1, 10, 50],
        "clf__max_features": ["sqrt", 0.5, 1.0],
        "clf__max_samples": [0.1, 0.3],
    },
    "HistGBM": {
        "clf__learning_rate": [0.05, 0.1, 0.2],
        "clf__max_iter": [200, 400, 600],
        "clf__max_leaf_nodes": [15, 31, 63],
        "clf__min_samples_leaf": [20, 50, 100],
        "clf__l2_regularization": [0.0, 0.1, 1.0],
    },
}

param_dist = espacios_busqueda[nombre_ganador]
print(f"Optimizando {nombre_ganador} — {len(param_dist)} hiperparámetros en el espacio")
for k, v in param_dist.items():
    print(f"  {k}: {v}")

Lanzamos la búsqueda. `n_iter=30` combinaciones aleatorias × `cv=3` = 90 entrenamientos sobre la muestra de 2M. `scoring='f1'` para que la búsqueda optimice la misma métrica que usamos para seleccionar el ganador. `n_jobs=-1` paraleliza los folds y combinaciones independientes. `verbose=1` imprime el progreso sin ahogar la salida.

In [ ]:
busqueda = RandomizedSearchCV(
    pipeline_ganador_base,
    param_distributions=param_dist,
    n_iter=30,
    cv=3,
    scoring="f1",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1,
    refit=False,
)

t0 = time.perf_counter()
busqueda.fit(X_muestra, y_muestra)
t_busqueda = time.perf_counter() - t0
print(f"\nBúsqueda completada en {t_busqueda / 60:.1f} min")
print(f"Mejor F1 en la muestra (CV): {busqueda.best_score_:.4f}")
print(f"Mejores hiperparámetros:")
for k, v in busqueda.best_params_.items():
    print(f"  {k} = {v}")

Re-entrenamos el pipeline ganador con los mejores hiperparámetros sobre el **train completo**, no sobre la muestra. Esto es crucial: la búsqueda sirve para encontrar la configuración, pero el modelo final tiene que haber visto todos los datos disponibles.

In [ ]:
pipeline_optimizado = pipeline_ganador_base.set_params(**busqueda.best_params_)

t0 = time.perf_counter()
pipeline_optimizado.fit(X_train, y_train)
t_refit = time.perf_counter() - t0
print(f"Refit sobre el train completo en {t_refit:.1f} s")

Evaluamos el modelo optimizado en validación y comparamos con el base del mismo tipo. Lo que esperamos es una mejora moderada en F1 — si no hay mejora o es marginal, significa que los defaults ya eran buenos y la búsqueda confirma que no hay mucho margen.

In [ ]:
resultados_optimizado = evaluar(pipeline_optimizado, X_val, y_val, f"{nombre_ganador}_opt", t_refit)

comparacion = pd.DataFrame([
    {"modelo": f"{nombre_ganador} (base)", **{k: v for k, v in tabla_base.loc[nombre_ganador].items()}},
    {"modelo": f"{nombre_ganador} (optimizado)", **{k: v for k, v in resultados_optimizado.items() if k != "modelo"}},
]).set_index("modelo")
comparacion

# 7. Interpretabilidad del modelo

Antes de guardar el modelo miramos qué features considera importantes. No es solo un ejercicio académico: si una feature que no debería importar aparece en primera posición, es señal de fuga de datos o de que algo raro está pasando. Y al revés, si las features que esperábamos (hora punta, tipo de vía) quedan relegadas, el modelo no está aprendiendo la lógica del problema y quizá necesita más features o más datos.

Usamos `permutation_importance`, que es agnóstico al modelo: mezcla aleatoriamente los valores de una feature y mide cuánto cae el F1. Una caída grande significa que el modelo dependía mucho de esa feature. Lo calculamos sobre una **muestra de 100.000 filas de val**, porque permutation_importance es caro (reentrena predicciones × número de features × repeticiones) y con 2.5M de filas tardaría demasiado.


In [ ]:
X_val_muestra, _, y_val_muestra, _ = train_test_split(
    X_val,
    y_val,
    train_size=min(TAM_MUESTRA_IMPORTANCIAS, len(X_val)),
    stratify=y_val,
    random_state=RANDOM_STATE,
)

importancias = permutation_importance(
    pipeline_optimizado,
    X_val_muestra,
    y_val_muestra,
    scoring="f1",
    n_repeats=5,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

df_imp = pd.DataFrame({
    "feature": X_val_muestra.columns,
    "importancia": importancias.importances_mean,
    "std": importancias.importances_std,
}).sort_values("importancia", ascending=False)
df_imp

Gráfica de barras horizontales con las features ordenadas por importancia. Incluimos la desviación estándar como barra de error para ver cuán estable es la medida entre las 5 repeticiones.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
orden = df_imp.sort_values("importancia")
ax.barh(
    orden["feature"],
    orden["importancia"],
    xerr=orden["std"],
    color=COLOR_AZUL,
    edgecolor="none",
)
ax.set_title(f"Importancia por permutación — {nombre_ganador} optimizado")
ax.set_xlabel("Caída en F1 al permutar la feature")
ax.axvline(0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

Para interpretar estos resultados pensamos en qué esperábamos a priori y qué vemos.

Lo esperable es que las variables temporales (`hora`, `es_hora_punta`, `dia_semana`, `es_laborable`) aparezcan altas, porque el NB01 mostró que la congestión sigue un patrón muy marcado por hora y día de la semana: hay picos a las 8-9 de la mañana y a las 18-20, y los fines de semana tienen menos tráfico. Si el modelo no estuviera capturando esto, no estaría aprendiendo la lógica básica del problema.

`tipo_elem` también debería pesar, porque M30 y urbano son dos regímenes de tráfico muy distintos (umbrales de congestión diferentes en el propio NB02 — 50 para urbano, 70 para M30), así que saber la familia del sensor cambia las reglas.

Las coordenadas `utm_x` y `utm_y` y `distrito` aportan señal geográfica: zonas céntricas tienen más atascos que la periferia. Lo que se confirme exactamente depende de cómo el modelo haya aprendido a combinar estas variables.

Si hay alguna feature que esperábamos importante y queda muy baja, o alguna que no debería y sale alta, hay que volver a revisarla.


# 8. Exportación para la evaluación final

Guardamos el pipeline optimizado con `joblib.dump`. El archivo incluye todo: el `ColumnTransformer` con sus transformaciones, el modelo entrenado con sus pesos, y toda la configuración. El NB04 solo tiene que cargar el joblib y llamar a `predict` o `predict_proba` sobre X_test o X_eval_final — no necesita reproducir ninguna transformación manualmente.

Junto al modelo guardamos un JSON con los metadatos relevantes (nombre del modelo, mejores hiperparámetros, F1 en val, fecha de entrenamiento) para tener trazabilidad de qué modelo es cada joblib si llegamos a entrenar varios.


In [ ]:
joblib.dump(pipeline_optimizado, RUTA_MODELO)
print(f"Modelo guardado en {RUTA_MODELO}")

metadatos = {
    "nombre_modelo": nombre_ganador,
    "mejores_hiperparametros": busqueda.best_params_,
    "f1_cv_muestra": float(busqueda.best_score_),
    "f1_val": float(resultados_optimizado["f1"]),
    "precision_val": float(resultados_optimizado["precision"]),
    "recall_val": float(resultados_optimizado["recall"]),
    "pr_auc_val": float(resultados_optimizado["pr_auc"]),
    "features": list(X_train.columns),
    "n_train": int(len(X_train)),
    "tam_muestra_busqueda": int(len(X_muestra)),
}
with RUTA_METADATOS.open("w", encoding="utf-8") as f:
    json.dump(metadatos, f, indent=2, ensure_ascii=False, default=str)
print(f"Metadatos guardados en {RUTA_METADATOS}")

Prueba rápida de que el joblib se carga correctamente y predice. Si esto falla, el NB04 tampoco podrá usarlo.

In [ ]:
modelo_recargado = joblib.load(RUTA_MODELO)
proba_prueba = modelo_recargado.predict_proba(X_val.head(5))
print("Probabilidades de las 5 primeras filas de val:")
print(proba_prueba)
assert proba_prueba.shape == (5, 2)
assert ((proba_prueba >= 0) & (proba_prueba <= 1)).all()
print("\nRecarga OK.")

## Resumen

Hemos entrenado tres modelos base, comparado sus métricas en validación, optimizado los hiperparámetros del mejor y guardado el pipeline final en `data/final/modelo_congestion.joblib`.

Lo que queda para el NB04:

- Evaluación profunda sobre `X_test` y sobre el holdout `X_eval_final` (que representa el mes más reciente, no visto durante el entrenamiento ni la selección).
- Matriz de confusión, curvas ROC y precision-recall.
- Análisis de errores por hora, día de la semana, distrito y tipo de sensor, para entender dónde se equivoca el modelo.
- Demo práctica: dado un sensor y un momento, qué probabilidad de congestión predice.

Para que el NB04 pueda cargar el modelo tiene que tener definida la función `codificacion_ciclica` antes del `joblib.load`, porque el pipeline la referencia internamente.
